# Module 6: Deep Learning Forecasting with PyTorch

This notebook demonstrates:
- Preparing sequences for LSTM/GRU
- Training deep learning models
- Generating multi-step forecasts
- Comparing DL vs baselines

**Prerequisites**: Install PyTorch: `pip install torch`


In [1]:
#!pip install torch

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import sys
warnings.filterwarnings('ignore')

sys.path.append('..')

# Check PyTorch
try:
    import torch
    print(f"✓ PyTorch {torch.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("❌ PyTorch not found in this kernel's Python environment.")
    print("\nTo install PyTorch in the notebook's kernel, run this cell:")
    print("  !pip install torch")
    print("\nOr install from terminal (ensure you're using the same Python as Jupyter):")
    print("  pip install torch")
    print("\nThen restart the kernel (Kernel → Restart Kernel) and re-run this cell.")
    print(f"\nCurrent Python: {sys.executable}")

from src.data.loaders import load_sales_data
from src.models.baseline import moving_average_forecast
from src.evaluation.metrics import mae, rmse, wape

if TORCH_AVAILABLE:
    from src.models.dl_models import lstm_forecast, gru_forecast
    print('✓ All imports OK')
else:
    print('\n⚠️  PyTorch not available. DL functions will not work.')
    print('   Install PyTorch and restart kernel to continue.')


✓ PyTorch 2.9.1+cpu
✓ CUDA available: False
✓ All imports OK


## Install PyTorch (if needed)

If PyTorch wasn't found above, uncomment and run the cell below to install it in this kernel's environment.

**Note**: After installing, you'll need to **restart the kernel** (Kernel → Restart Kernel) and re-run all cells.


In [3]:
# Uncomment the line below if PyTorch is not installed
# !pip install torch

# After running, restart the kernel (Kernel → Restart Kernel) and re-run from the top


## Load Data and Select SKU

We'll pick a SKU with sufficient history (200+ observations) for deep learning.


In [4]:
raw_path = Path('../data/raw/sample_sales.csv')
df = load_sales_data(raw_path)

# Find SKU with most history
sku_counts = df.groupby('sku_id').size().sort_values(ascending=False)
top_sku = sku_counts.index[0]
print(f"Selected SKU: {top_sku} ({sku_counts[top_sku]} observations)")

df_sku = df[df['sku_id'] == top_sku].sort_values('date')
df_sku['date'] = pd.to_datetime(df_sku['date'])

# Split train/test (use last 30 days as test)
cutoff = df_sku['date'].max() - pd.Timedelta(days=30)
train = df_sku[df_sku['date'] <= cutoff]
test = df_sku[df_sku['date'] > cutoff]

print(f"Train: {len(train)} observations ({train['date'].min()} to {train['date'].max()})")
print(f"Test: {len(test)} observations ({test['date'].min()} to {test['date'].max()})")

y_train = train['units_sold'].values.astype(float)
y_test = test['units_sold'].values.astype(float)

# Normalize (min-max scaling)
train_min, train_max = y_train.min(), y_train.max()
y_train_norm = (y_train - train_min) / (train_max - train_min + 1e-8)

print(f"\nTrain stats: min={y_train.min():.1f}, max={y_train.max():.1f}, mean={y_train.mean():.1f}")


Selected SKU: SKU001 (730 observations)
Train: 700 observations (2023-12-18 00:00:00 to 2025-11-16 00:00:00)
Test: 30 observations (2025-11-17 00:00:00 to 2025-12-16 00:00:00)

Train stats: min=0.0, max=49.0, mean=15.7


## Train LSTM Model


In [5]:
horizon = len(y_test)  # Forecast all test days
seq_len = 30

if not TORCH_AVAILABLE:
    print("⚠️  PyTorch not available. Skipping LSTM training.")
    print("   Install PyTorch (see cell above) and restart kernel.")
    fc_lstm = None
else:
    print(f"Training LSTM (seq_len={seq_len}, horizon={horizon})...")
    result_lstm = lstm_forecast(
        y_train=y_train_norm,
        horizon=horizon,
        seq_len=seq_len,
        hidden_size=64,
        num_layers=2,
        epochs=50,
        batch_size=32,
        lr=0.001,
        verbose=True
    )
    
    # Denormalize
    fc_lstm = result_lstm.y_pred * (train_max - train_min) + train_min
    print(f"\nLSTM forecast range: [{fc_lstm.min():.1f}, {fc_lstm.max():.1f}]")


Training LSTM (seq_len=30, horizon=30)...
Epoch 10/50, Loss: 0.0351
Epoch 20/50, Loss: 0.0351
Epoch 30/50, Loss: 0.0353
Epoch 40/50, Loss: 0.0355
Epoch 50/50, Loss: 0.0356

LSTM forecast range: [15.0, 15.3]


## Train GRU Model (Optional - faster alternative)


In [6]:
if not TORCH_AVAILABLE:
    print("⚠️  PyTorch not available. Skipping GRU training.")
    fc_gru = None
else:
    print(f"Training GRU (seq_len={seq_len}, horizon={horizon})...")
    result_gru = gru_forecast(
        y_train=y_train_norm,
        horizon=horizon,
        seq_len=seq_len,
        hidden_size=64,
        num_layers=2,
        epochs=50,
        batch_size=32,
        lr=0.001,
        verbose=True
    )
    
    # Denormalize
    fc_gru = result_gru.y_pred * (train_max - train_min) + train_min
    print(f"\nGRU forecast range: [{fc_gru.min():.1f}, {fc_gru.max():.1f}]")


Training GRU (seq_len=30, horizon=30)...
Epoch 10/50, Loss: 0.0361
Epoch 20/50, Loss: 0.0348
Epoch 30/50, Loss: 0.0346
Epoch 40/50, Loss: 0.0352
Epoch 50/50, Loss: 0.0347

GRU forecast range: [14.6, 16.7]


## Compare with Baseline


In [7]:
# Baseline: moving average
result_ma = moving_average_forecast(y_train, horizon=horizon, window=7)
fc_ma = result_ma.y_pred

# Evaluate
metrics = {
    'Moving Average': {
        'MAE': mae(y_test, fc_ma),
        'RMSE': rmse(y_test, fc_ma),
        'WAPE': wape(y_test, fc_ma),
    },
}

if TORCH_AVAILABLE and fc_lstm is not None:
    metrics['LSTM'] = {
        'MAE': mae(y_test, fc_lstm),
        'RMSE': rmse(y_test, fc_lstm),
        'WAPE': wape(y_test, fc_lstm),
    }

if TORCH_AVAILABLE and fc_gru is not None:
    metrics['GRU'] = {
        'MAE': mae(y_test, fc_gru),
        'RMSE': rmse(y_test, fc_gru),
        'WAPE': wape(y_test, fc_gru),
    }

comparison = pd.DataFrame(metrics).T
print(comparison.round(3))


                   MAE    RMSE   WAPE
Moving Average  12.052  15.336  0.592
LSTM            13.368  15.694  0.656
GRU             13.083  15.476  0.642


## Visualize Forecasts


In [ ]:
plt.figure(figsize=(14, 6))

# Plot last 60 days of training + test period
plot_train = train.tail(60)
plot_test = test

plt.plot(plot_train['date'], plot_train['units_sold'], 'b-', label='Train (last 60 days)', alpha=0.7)
plt.plot(plot_test['date'], plot_test['units_sold'], 'k-', label='Actual', linewidth=2)

if TORCH_AVAILABLE and fc_lstm is not None:
    plt.plot(plot_test['date'], fc_lstm, 'r--', label='LSTM Forecast', linewidth=2)
if TORCH_AVAILABLE and fc_gru is not None:
    plt.plot(plot_test['date'], fc_gru, 'g--', label='GRU Forecast', linewidth=2)

plt.plot(plot_test['date'], fc_ma, 'm--', label='Moving Avg Forecast', linewidth=2)

plt.axvline(x=cutoff, color='gray', linestyle=':', label='Cutoff')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.title(f'Forecast Comparison: {top_sku}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
